In [3]:
# pip install ruptures

In [10]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import ruptures as rpt
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# Full GGS-like Robust Change Point Detection
# ------------------------------------------------------------
# Input  : ./tvpvar_input_scaled.csv
# Output : ./result_full_robust/
#
# Method:
# - Full market-wide regime detection
# - min_size = 30, 60, 90, 120 robustness check
# - PELT-based candidate generation
# - Gaussian NLL + BIC-style criterion for model selection
# - No arbitrary target cp count heuristic
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")
RESULT_DIR = BASE_DIR / "result_full_robust"
PLOT_DIR = RESULT_DIR / "plots"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

DATE_COL = "Date"

FEATURES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
    "d_PJM_LMP",
]

# Robustness check
MIN_SIZE_LIST = [30, 60, 90, 120]

# PELT penalty candidates
PENALTY_LIST = [5, 7, 10, 15, 20, 30, 50]

# normal: Gaussian regime assumption
# rbf: robustness candidate
MODEL_LIST = ["normal", "rbf"]

# ±WINDOW_DAYS 이내 change point를 같은 structural date cluster로 묶음
WINDOW_DAYS = 10

EPS = 1e-8

# -----------------------------
# Output files
# -----------------------------
OUT_GRID = RESULT_DIR / "full_ggs_grid_search_bic_by_minsize.csv"
OUT_SELECTED = RESULT_DIR / "full_ggs_selected_by_minsize.csv"
OUT_CP = RESULT_DIR / "full_ggs_change_points_by_minsize.csv"
OUT_SEGMENTS = RESULT_DIR / "full_ggs_segments_by_minsize.csv"
OUT_STATS = RESULT_DIR / "full_ggs_segment_stats_by_minsize.csv"
OUT_ROBUST = RESULT_DIR / "full_ggs_robust_change_point_clusters.csv"
OUT_PLOT_COMBINED = PLOT_DIR / "full_ggs_robust_change_points_all_features.png"


# ============================================================
# Data
# ============================================================
def validate_input(df):
    if DATE_COL not in df.columns:
        raise ValueError(f"'{DATE_COL}' column not found.")

    missing_cols = [c for c in FEATURES if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing feature columns: {missing_cols}")


def load_data():
    df = pd.read_csv(INPUT_FILE)
    validate_input(df)

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    data = df[[DATE_COL] + FEATURES].copy()
    data = data.replace([np.inf, -np.inf], np.nan)
    data = data.dropna(how="any")
    data = data.sort_values(DATE_COL).reset_index(drop=True)

    if data.empty:
        raise ValueError("No usable data after cleaning.")

    return data


# ============================================================
# Segmentation
# ============================================================
def run_pelt(X, model_name, penalty, min_size):
    try:
        algo = rpt.Pelt(
            model=model_name,
            min_size=min_size,
        ).fit(X)

        cps = algo.predict(pen=penalty)

        # ruptures returns len(X) as final boundary
        change_idx = [int(cp) for cp in cps if cp < len(X)]

        return change_idx, ""

    except Exception as e:
        return [], repr(e)


def get_bounds(change_idx, n_obs):
    return [0] + list(change_idx) + [n_obs]


def get_segment_lengths(change_idx, n_obs):
    bounds = get_bounds(change_idx, n_obs)

    return [
        bounds[i + 1] - bounds[i]
        for i in range(len(bounds) - 1)
    ]


# ============================================================
# Gaussian NLL and BIC-style criterion
# ============================================================
def gaussian_segment_nll(X_seg):
    """
    Multivariate Gaussian negative log-likelihood up to constants.

    NLL = n/2 * log|Sigma|
        + 1/2 * sum Mahalanobis distance
    """
    X_seg = np.asarray(X_seg, dtype=float)

    n, d = X_seg.shape

    if n <= d + 1:
        return np.inf

    mu = X_seg.mean(axis=0)
    Xc = X_seg - mu

    Sigma = (Xc.T @ Xc) / n
    Sigma = Sigma + np.eye(d) * EPS

    sign, logdet = np.linalg.slogdet(Sigma)

    if sign <= 0 or not np.isfinite(logdet):
        return np.inf

    inv_Sigma = np.linalg.inv(Sigma)

    mahal = np.einsum(
        "ij,jk,ik->i",
        Xc,
        inv_Sigma,
        Xc,
    ).sum()

    nll = 0.5 * n * logdet + 0.5 * mahal

    return float(nll)


def total_gaussian_nll(X, change_idx):
    bounds = get_bounds(change_idx, len(X))

    total = 0.0

    for i in range(len(bounds) - 1):
        start = bounds[i]
        end = bounds[i + 1]

        nll = gaussian_segment_nll(X[start:end])

        if not np.isfinite(nll):
            return np.inf

        total += nll

    return float(total)


def bic_style_score(X, change_idx):
    """
    BIC-style score:
    score = 2 * NLL + q * log(n)

    q = total Gaussian parameters across all segments
      = K * [d mean parameters + d(d+1)/2 covariance parameters]

    K = number of segments = number of change points + 1
    """
    n, d = X.shape
    K = len(change_idx) + 1

    nll = total_gaussian_nll(X, change_idx)

    n_params_per_segment = d + (d * (d + 1)) / 2
    q = K * n_params_per_segment

    score = 2.0 * nll + q * np.log(n)

    return float(score), float(nll), int(q)


def choose_best_candidate_for_minsize(grid_df, min_size):
    valid = grid_df[
        (grid_df["min_size"] == min_size) &
        (grid_df["error_msg"] == "") &
        np.isfinite(grid_df["bic_score"])
    ].copy()

    if valid.empty:
        raise RuntimeError(f"No valid candidate for min_size={min_size}")

    valid = valid.sort_values(
        ["bic_score", "n_cp", "min_segment_length"],
        ascending=[True, True, False],
    ).reset_index(drop=True)

    return valid.iloc[0]


# ============================================================
# Tables
# ============================================================
def make_change_point_table(min_size, model_name, penalty, change_idx, dates):
    return pd.DataFrame({
        "min_size": min_size,
        "model": model_name,
        "penalty": penalty,
        "cp_id": np.arange(1, len(change_idx) + 1),
        "change_idx": change_idx,
        "change_date": dates.iloc[change_idx].values,
    })


def make_segment_table(min_size, model_name, penalty, change_idx, data):
    dates = data[DATE_COL]
    bounds = get_bounds(change_idx, len(data))

    rows = []

    for seg_id in range(len(bounds) - 1):
        start_idx = bounds[seg_id]
        end_exclusive = bounds[seg_id + 1]
        end_idx = end_exclusive - 1

        rows.append({
            "min_size": min_size,
            "model": model_name,
            "penalty": penalty,
            "segment_id": seg_id,
            "start_idx": start_idx,
            "end_idx": end_idx,
            "start_date": dates.iloc[start_idx],
            "end_date": dates.iloc[end_idx],
            "length": end_exclusive - start_idx,
        })

    return pd.DataFrame(rows)


def make_segment_stats(min_size, model_name, penalty, data, segments_df):
    rows = []

    for _, seg in segments_df.iterrows():
        seg_id = int(seg["segment_id"])
        start_idx = int(seg["start_idx"])
        end_idx = int(seg["end_idx"])

        seg_data = data.iloc[start_idx:end_idx + 1]

        for feature in FEATURES:
            values = seg_data[feature].values

            rows.append({
                "min_size": min_size,
                "model": model_name,
                "penalty": penalty,
                "segment_id": seg_id,
                "feature": feature,
                "mean": float(np.mean(values)),
                "std": float(np.std(values, ddof=0)),
                "var": float(np.var(values, ddof=0)),
                "min": float(np.min(values)),
                "max": float(np.max(values)),
                "median": float(np.median(values)),
            })

    return pd.DataFrame(rows)


# ============================================================
# Robust cluster
# ============================================================
def make_robust_cp_clusters(cp_df, window_days=10):
    """
    여러 min_size에서 나온 change point를 날짜 기준으로 clustering.
    window_days 이내 날짜는 같은 structural date cluster로 간주.
    """
    if cp_df.empty:
        return pd.DataFrame(columns=[
            "cluster_id",
            "cluster_start",
            "cluster_end",
            "representative_date",
            "n_occurrences",
            "n_unique_min_size",
            "min_sizes",
            "dates",
        ])

    temp = cp_df.copy()
    temp["change_date"] = pd.to_datetime(temp["change_date"])
    temp = temp.sort_values("change_date").reset_index(drop=True)

    clusters = []
    current = [temp.iloc[0]]

    for i in range(1, len(temp)):
        prev_date = pd.to_datetime(current[-1]["change_date"])
        curr_date = pd.to_datetime(temp.iloc[i]["change_date"])

        if (curr_date - prev_date).days <= window_days:
            current.append(temp.iloc[i])
        else:
            clusters.append(current)
            current = [temp.iloc[i]]

    clusters.append(current)

    rows = []

    for cid, cluster in enumerate(clusters, start=1):
        cdf = pd.DataFrame(cluster)
        cdf["change_date"] = pd.to_datetime(cdf["change_date"])

        rep_date = cdf["change_date"].median()

        rows.append({
            "cluster_id": cid,
            "cluster_start": cdf["change_date"].min(),
            "cluster_end": cdf["change_date"].max(),
            "representative_date": rep_date,
            "n_occurrences": len(cdf),
            "n_unique_min_size": cdf["min_size"].nunique(),
            "min_sizes": "|".join(map(str, sorted(cdf["min_size"].unique()))),
            "dates": "|".join(cdf["change_date"].astype(str).tolist()),
        })

    return pd.DataFrame(rows)


# ============================================================
# Plot
# ============================================================
def plot_robust_full(data, cp_df, robust_df, out_path):
    dates = data[DATE_COL]

    plt.figure(figsize=(16, 7))

    for feature in FEATURES:
        plt.plot(
            dates,
            data[feature],
            linewidth=0.8,
            alpha=0.45,
            label=feature,
        )

    # 개별 min_size change points
    if not cp_df.empty:
        for _, row in cp_df.iterrows():
            plt.axvline(
                pd.to_datetime(row["change_date"]),
                linestyle="--",
                linewidth=0.8,
                alpha=0.35,
            )

    # robust cluster representative dates
    if not robust_df.empty:
        for _, row in robust_df.iterrows():
            plt.axvline(
                pd.to_datetime(row["representative_date"]),
                linestyle="-",
                linewidth=2.0,
                alpha=0.9,
            )

    plt.title("Full GGS-like Robust Change Points across min_size")
    plt.xlabel("Date")
    plt.ylabel("Standardized value")
    plt.legend(loc="upper left", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()


# ============================================================
# Main
# ============================================================
def main():
    print("================================================")
    print("Full GGS-like Robust Change Point Detection")
    print("min_size robustness: 30, 60, 90, 120")
    print("================================================")

    data = load_data()
    X = data[FEATURES].values
    dates = data[DATE_COL]

    print("Input file :", INPUT_FILE)
    print("Rows       :", len(data))
    print("Features   :", len(FEATURES))
    print("Date range :", dates.min(), "~", dates.max())
    print()

    # --------------------------------------------------------
    # Grid search across all min_size
    # --------------------------------------------------------
    grid_rows = []

    print("[1] Running full grid search...")

    for model_name in MODEL_LIST:
        for min_size in MIN_SIZE_LIST:
            for penalty in PENALTY_LIST:
                change_idx, error_msg = run_pelt(
                    X=X,
                    model_name=model_name,
                    penalty=penalty,
                    min_size=min_size,
                )

                seg_lengths = get_segment_lengths(change_idx, len(X))

                bic_score, nll, n_params = bic_style_score(
                    X,
                    change_idx,
                )

                grid_rows.append({
                    "model": model_name,
                    "min_size": min_size,
                    "penalty": penalty,
                    "n_cp": len(change_idx),
                    "n_segments": len(change_idx) + 1,
                    "min_segment_length": min(seg_lengths),
                    "max_segment_length": max(seg_lengths),
                    "mean_segment_length": float(np.mean(seg_lengths)),
                    "gaussian_nll": nll,
                    "n_params": n_params,
                    "bic_score": bic_score,
                    "change_idx": "|".join(map(str, change_idx)),
                    "change_dates": "|".join(
                        dates.iloc[change_idx].astype(str).tolist()
                    ) if len(change_idx) > 0 else "",
                    "error_msg": error_msg,
                })

    grid_df = pd.DataFrame(grid_rows)
    grid_df.to_csv(OUT_GRID, index=False, encoding="utf-8-sig")

    print("Saved grid search:", OUT_GRID)

    # --------------------------------------------------------
    # Select best segmentation for each min_size
    # --------------------------------------------------------
    selected_rows = []
    cp_tables = []
    segment_tables = []
    stat_tables = []

    print()
    print("[2] Selecting best candidate for each min_size...")

    for min_size in MIN_SIZE_LIST:
        best = choose_best_candidate_for_minsize(
            grid_df,
            min_size=min_size,
        )

        best_model = best["model"]
        best_penalty = float(best["penalty"])

        change_idx, error_msg = run_pelt(
            X=X,
            model_name=best_model,
            penalty=best_penalty,
            min_size=min_size,
        )

        if error_msg:
            raise RuntimeError(
                f"Best candidate failed for min_size={min_size}: {error_msg}"
            )

        seg_lengths = get_segment_lengths(change_idx, len(X))
        bic_score, nll, n_params = bic_style_score(X, change_idx)

        selected_rows.append({
            "min_size": min_size,
            "selected_model": best_model,
            "selected_penalty": best_penalty,
            "n_cp": len(change_idx),
            "n_segments": len(change_idx) + 1,
            "min_segment_length": min(seg_lengths),
            "max_segment_length": max(seg_lengths),
            "mean_segment_length": float(np.mean(seg_lengths)),
            "gaussian_nll": nll,
            "n_params": n_params,
            "bic_score": bic_score,
            "change_idx": "|".join(map(str, change_idx)),
            "change_dates": "|".join(
                dates.iloc[change_idx].astype(str).tolist()
            ) if len(change_idx) > 0 else "",
        })

        cp_tables.append(
            make_change_point_table(
                min_size=min_size,
                model_name=best_model,
                penalty=best_penalty,
                change_idx=change_idx,
                dates=dates,
            )
        )

        seg_df = make_segment_table(
            min_size=min_size,
            model_name=best_model,
            penalty=best_penalty,
            change_idx=change_idx,
            data=data,
        )

        segment_tables.append(seg_df)

        stat_tables.append(
            make_segment_stats(
                min_size=min_size,
                model_name=best_model,
                penalty=best_penalty,
                data=data,
                segments_df=seg_df,
            )
        )

        print()
        print(f"min_size = {min_size}")
        print("model           :", best_model)
        print("penalty         :", best_penalty)
        print("n_cp            :", len(change_idx))
        print("segment lengths :", seg_lengths)
        print("BIC-style score :", round(bic_score, 4))
        print("change idx      :", change_idx)

    selected_df = pd.DataFrame(selected_rows)
    cp_df = pd.concat(cp_tables, ignore_index=True) if cp_tables else pd.DataFrame()
    segments_df = pd.concat(segment_tables, ignore_index=True) if segment_tables else pd.DataFrame()
    stats_df = pd.concat(stat_tables, ignore_index=True) if stat_tables else pd.DataFrame()

    selected_df.to_csv(OUT_SELECTED, index=False, encoding="utf-8-sig")
    cp_df.to_csv(OUT_CP, index=False, encoding="utf-8-sig")
    segments_df.to_csv(OUT_SEGMENTS, index=False, encoding="utf-8-sig")
    stats_df.to_csv(OUT_STATS, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Robust change point clustering
    # --------------------------------------------------------
    robust_df = make_robust_cp_clusters(
        cp_df,
        window_days=WINDOW_DAYS,
    )

    robust_df.to_csv(OUT_ROBUST, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    plot_robust_full(
        data=data,
        cp_df=cp_df,
        robust_df=robust_df,
        out_path=OUT_PLOT_COMBINED,
    )

    # --------------------------------------------------------
    # Print summary
    # --------------------------------------------------------
    print()
    print("[3] Selected by min_size")
    print(selected_df.to_string(index=False))

    print()
    print("[4] Change point table")
    if cp_df.empty:
        print("No change points detected.")
    else:
        print(cp_df.to_string(index=False))

    print()
    print("[5] Robust change point clusters")
    if robust_df.empty:
        print("No robust clusters.")
    else:
        print(robust_df.to_string(index=False))

    print()
    print("Saved files:")
    print(" -", OUT_GRID)
    print(" -", OUT_SELECTED)
    print(" -", OUT_CP)
    print(" -", OUT_SEGMENTS)
    print(" -", OUT_STATS)
    print(" -", OUT_ROBUST)
    print(" -", OUT_PLOT_COMBINED)

    print()
    print("Finished.")


if __name__ == "__main__":
    main()

Full GGS-like Robust Change Point Detection
min_size robustness: 30, 60, 90, 120
Input file : tvpvar_input_scaled.csv
Rows       : 606
Features   : 10
Date range : 2022-12-01 00:00:00 ~ 2026-01-06 00:00:00

[1] Running full grid search...
Saved grid search: result_full_robust\full_ggs_grid_search_bic_by_minsize.csv

[2] Selecting best candidate for each min_size...

min_size = 30
model           : rbf
penalty         : 5.0
n_cp            : 0
segment lengths : [606]
BIC-style score : 5608.3146
change idx      : []

min_size = 60
model           : rbf
penalty         : 5.0
n_cp            : 0
segment lengths : [606]
BIC-style score : 5608.3146
change idx      : []

min_size = 90
model           : rbf
penalty         : 5.0
n_cp            : 0
segment lengths : [606]
BIC-style score : 5608.3146
change idx      : []

min_size = 120
model           : normal
penalty         : 30.0
n_cp            : 3
segment lengths : [180, 140, 145, 141]
BIC-style score : 5412.4527
change idx      : [180, 3